# P2｜BEATs Native Encoder

**状态：Preflight / Adapter HOLD；首轮 core。** 目的：与 P1 完全匹配，只把四数据集共享的 frozen candidate encoder package 从 AST 换为 BEATs。

## 唯一变量、匹配对照与四数据集边界

matched comparator=P1。四数据集都使用相同 2.0 s / 1.0 s source-time windows、同一个 frozen BEATs、同一个 trainable Linear 768→256 projector 与 dataset-native heads。除 AST+frontend package→BEATs+frontend package 外，window、split/group、head、sampler、scope、budget、seed、selection 与 metrics 必须完全匹配。

ICBHI/SPRSound/KAUH 对 valid window embeddings masked mean；HF 保留 projected sequence [B,K,256] 并输出 I/E/CAS/DAS [B,K,4]。这是四数据集 shared encoder/projector，但不是统一标签任务。

- ICBHI cycle flat4 [B,4]；SPRSound event binary [B,2] 与 raw7 [B,7]；KAUH recording raw9 [B,9]，B/D/E 同 patient group。
- HF window-center target 使用 paper-native one-vs-rest rasterization；constructed negative 不等于 raw normal，missing/unknown 继续 fail closed。
- 输入/输出：16 kHz native units → windows [B,K,32000] → BEATs embeddings [B,K,768] → shared projected [B,K,256] → native aggregation/heads。P2−P1 只能解释为 encoder+frontend package replacement。

In [ ]:
from pathlib import Path
import os
from baseline.multidataset_pipeline.preflight import P1_P5_SELECTION_RULE, P1_P5_UPDATE_BUDGET, freeze_receipt

PIPELINE = {
    "id": "P2",
    "comparator": "P1",
    "only_change": "four_dataset_shared_encoder_package: AST_to_BEATs",
    "seed": 20260728,
    "split_policy": "reuse_P1_immutable_receipts",
    "encoder_scope": "frozen_pretrained_BEATs",
    "shared_projector": "minimal_linear_projector_768_to_256_match_P1",
    "shared_projector_bias": True,
    "shared_projector_lanes": ["ICBHI", "SPRSound", "HF", "KAUH"],
    "window_policy": "source_time_2s_window_1s_stride_match_P1",
    "sampler": "four_dataset_source_proportional_match_P1",
    "hf_lane": "shared_projected_window_sequence_to_temporal4_head",
    "hf_uses_shared_projector": True,
    "trainable_scope": "one_shared_projector_768_to_256_plus_four_dataset_native_heads_match_P1",
    "contract_modules": ["baseline.multidataset_pipeline.contracts", "baseline.multidataset_pipeline.sliding_window", "baseline.multidataset_pipeline.joint_native", "baseline.multidataset_pipeline.preflight"],
    "engineering_test": "tests/test_multidataset_pipeline.py::JointNativeContractTest",
    "batch_size": 8,
    "update_budget": P1_P5_UPDATE_BUDGET,
    "validation_interval_updates": 1725,
    "selection": P1_P5_SELECTION_RULE,
    "output_dir": "result/reproduce/P2_beats_native_encoder",
    "receipt_path": "result/reproduce/P2_beats_native_encoder/P2_receipt.json",
}
PROJECT_ROOT = Path(os.environ.get("ACOUSTIC_PROJECT_ROOT", Path.cwd())).resolve()
APPROVAL_RECEIPT = os.environ.get("P2_APPROVAL_RECEIPT")


## 科学 gate 与运行前审批

P1–P5 共同冻结 seed=20260728、batch size=8、86250 updates、每 1725 updates 做 validation-only selection；不得报告跨数据集 pooled Score。BEATs checkpoint/source revision/SHA、2 s window exact mask、B*K flatten/unflatten lineage、short zero-padding invariance、HF window target counts 和 CUDA smoke 仍须闭合。配置 verifier 要证明除四数据集 shared encoder package 外零差异；缺失 approval/hash 时 fail closed。

In [ ]:
PREFLIGHT = freeze_receipt()
required = [PIPELINE["update_budget"], PIPELINE["selection"], APPROVAL_RECEIPT]
if any(value in (None, "") for value in required):
    raise RuntimeError("P2 fail closed: approval receipt is missing")
approval_path = PROJECT_ROOT / APPROVAL_RECEIPT
if not approval_path.is_file():
    raise FileNotFoundError(approval_path)
DRY_RUN_PLAN = {"pipeline": PIPELINE, "approval": str(approval_path), "execute": False}


## 输出、receipt 与结果表

receipt 增加 P1 parity hash、BEATs checkpoint/frontend/window-adapter hashes、per-dataset unit/window/valid counts、HF target semantics、native metrics 与 independent verifier。结果必须按任务报告，不能 pooled。

| Comparison | Native task | Result | Decision |
|---|---|---:|---|
| P2−P1 | ICBHI / SPRSound / HF / KAUH native tasks | Not run | HOLD |

**Test Result = Not run。Decision = HOLD。Claim boundary：只能解释四数据集 shared-window encoder+frontend package 的 AST→BEATs 替换。**